# 📈 Notebook 05 — Model Evaluation & Comparison
> **Purpose:** Side-by-side comparison of all three models on classification
> and financial metrics. Produce the capstone results table.

**Inputs:** `data/baseline_results.csv`, `data/cnn_lstm_preds.npy`, model checkpoints  
**Outputs:** `data/final_results.csv`, comparison plots

---

## 5.1  Load all predictions

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import torch
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score, roc_auc_score
)
sns.set_theme(style='darkgrid')
%matplotlib inline

y_test_seq  = np.load('data/y_test_seq.npy')
y_pred_cnn  = np.load('data/cnn_lstm_preds.npy')
y_probs_cnn = np.load('data/cnn_lstm_probs.npy')
y_test_2d   = np.load('data/y_test_2d.npy')
X_test_2d   = np.load('data/X_test_2d.npy')
X_test_seq  = np.load('data/X_test_seq.npy')

In [ ]:
# XGBoost predictions
xgb_model = xgb.XGBClassifier()
xgb_model.load_model('data/xgb_model.json')
y_pred_xgb = xgb_model.predict(X_test_2d)

# LSTM-only predictions
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class LSTMOnly(torch.nn.Module):
    def __init__(self, input_size=42, hidden=128, num_layers=2, n_classes=3, dropout=0.3):
        super().__init__()
        self.lstm = torch.nn.LSTM(input_size, hidden, num_layers,
                                   batch_first=True, dropout=dropout)
        self.fc = torch.nn.Sequential(
            torch.nn.Linear(hidden, 64), torch.nn.ReLU(),
            torch.nn.Dropout(dropout),   torch.nn.Linear(64, n_classes)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

lstm_model = LSTMOnly().to(DEVICE)
lstm_model.load_state_dict(torch.load('data/lstm_only_model.pt', map_location=DEVICE))
lstm_model.eval()

Xt = torch.tensor(X_test_seq, dtype=torch.float32)
with torch.no_grad():
    y_pred_lstm = lstm_model(Xt.to(DEVICE)).argmax(1).cpu().numpy()

print('Predictions loaded for all 3 models.')

## 5.2  Classification metrics table

In [ ]:
def metrics(name, y_true, y_pred):
    return {
        'Model':    name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'F1-macro': f1_score(y_true, y_pred, average='macro'),
        'F1-DOWN':  f1_score(y_true, y_pred, average=None)[0],
        'F1-FLAT':  f1_score(y_true, y_pred, average=None)[1],
        'F1-UP':    f1_score(y_true, y_pred, average=None)[2],
    }

results = pd.DataFrame([
    metrics('XGBoost',   y_test_2d,  y_pred_xgb),
    metrics('LSTM-only', y_test_seq, y_pred_lstm),
    metrics('CNN-LSTM',  y_test_seq, y_pred_cnn),
]).set_index('Model').round(4)

display(results.style.background_gradient(cmap='RdYlGn', axis=0))

## 5.3  Confusion matrices — side by side

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
models   = ['XGBoost', 'LSTM-only', 'CNN-LSTM']
y_preds  = [y_pred_xgb, y_pred_lstm, y_pred_cnn]
y_trues  = [y_test_2d, y_test_seq, y_test_seq]

for ax, name, yt, yp in zip(axes, models, y_trues, y_preds):
    cm = confusion_matrix(yt, yp, normalize='true')
    sns.heatmap(cm, annot=True, fmt='.2%', cmap='Blues',
                xticklabels=['DOWN','FLAT','UP'],
                yticklabels=['DOWN','FLAT','UP'], ax=ax)
    ax.set_title(f'{name}\nAcc={accuracy_score(yt,yp):.3f}  F1={f1_score(yt,yp,average="macro"):.3f}')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — All Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('data/fig_confusion_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 5.4  Accuracy & F1 bar chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

model_names = results.index.tolist()
x = np.arange(len(model_names))
w = 0.35

axes[0].bar(x, results['Accuracy'], width=0.5,
            color=['steelblue','darkorange','seagreen'])
axes[0].axhline(0.60, ls='--', color='red', label='Target 60%')
axes[0].set_xticks(x); axes[0].set_xticklabels(model_names)
axes[0].set_title('Test Accuracy'); axes[0].set_ylim(0.45, 0.75)
axes[0].legend()

axes[1].bar(x, results['F1-macro'], width=0.5,
            color=['steelblue','darkorange','seagreen'])
axes[1].axhline(0.59, ls='--', color='red', label='Target F1=0.59')
axes[1].set_xticks(x); axes[1].set_xticklabels(model_names)
axes[1].set_title('F1-macro'); axes[1].set_ylim(0.45, 0.70)
axes[1].legend()

plt.suptitle('Model Comparison — Classification Metrics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('data/fig_model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 5.5  Hit ratio (directional accuracy — excl. FLAT predictions)

In [ ]:
def hit_ratio(y_true, y_pred):
    """Fraction of UP/DOWN predictions that were correct (excludes FLAT preds)."""
    mask = y_pred != 1    # exclude FLAT (1)
    if mask.sum() == 0:
        return np.nan
    return accuracy_score(y_true[mask], y_pred[mask])

hit_rows = []
for name, yt, yp in zip(models, y_trues, y_preds):
    hr = hit_ratio(yt, yp)
    flat_pct = (yp == 1).mean() * 100
    hit_rows.append({'Model': name, 'Hit Ratio': hr, 'Flat %': flat_pct})

hit_df = pd.DataFrame(hit_rows).set_index('Model').round(4)
display(hit_df)

## 5.6  Save final results

In [ ]:
final = results.copy()
final['Hit Ratio'] = [hit_ratio(yt, yp)
                       for yt, yp in zip(y_trues, y_preds)]
final.to_csv('data/final_results.csv')
print('Saved data/final_results.csv')
display(final.round(4))

---
> ✅ **Evaluation complete.** Proceed to `06_order_execution_optimizer.ipynb`.